In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_3")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}
n = 3
condition_pal = {
    "wt": blues[n],
    "bcd": reds[2],
    "trk": greens[2],
}

In [ ]:
import dnt
import pandas as pd
from collections import defaultdict

all_surface_areas = defaultdict(dict)
all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

    points = df[df["frame"] == min_mvmt_frames[-1]][["x", "y", "z"]].values
    aps = df[df["frame"] == min_mvmt_frames[-1]]["AP"].values
    mesh = dnt.mesh_from_points(points)

    ap_vals = np.linspace(0.001, 0.98, 6)
    y_max = points[:, 1].max()
    y_min = points[:, 1].min()
    y_vals = y_min + ap_vals * (y_max - y_min)

    try:
        surface_areas = dnt.calculate_surface_area_along_axis(mesh, y_vals)
    except Exception as e:
        mesh = dnt.smoothed_mesh_from_points(points)
        surface_areas = dnt.calculate_surface_area_along_axis(mesh, y_vals)

    # flip order of surface areas if y_max corresponds to AP=0
    if aps[np.argmax(points[:, 1])] < aps[np.argmin(points[:, 1])]:
        surface_areas = surface_areas[::-1]

    total_surface_area = surface_areas.sum()

    ap_bounds = [(l, r) for l, r in zip([0.0, *ap_vals], [*ap_vals, 1.0])]
    centers = [(l + r)/2 for l, r in ap_bounds]

    all_surface_areas[stem]["centers"] = centers
    all_surface_areas[stem]["areas"] = surface_areas
    all_surface_areas[stem]["bounds"] = ap_bounds

    all_surface_areas[stem] = pd.DataFrame(all_surface_areas[stem])

    spots_dfs[k]["AP_bin"] = pd.cut(
        spots_dfs[k]["AP"], bins=[0.0, *ap_vals, 1.0], labels=centers
    )

print(all_mmfs)

for stem, surface_area_df in all_surface_areas.items():
    surface_area_df["area_normed"] = surface_area_df["areas"] / surface_area_df["areas"].sum()
    surface_area_df["area_normed"] = surface_area_df["area_normed"].astype(float)
    surface_area_df["center_copy"] = surface_area_df["centers"]
    surface_area_df.set_index("center_copy", inplace=True)

cycle_relative_densities = defaultdict(list)
intermediate_densities = defaultdict(list)

for k, spots_df in enumerate(spots_dfs):
    stem = stems[k]
    print(stem)

    """
    Density related metrics
    """
    spots_df["nucleus_weight"] = 1 / spots_df["frame"].map(spots_df.groupby("frame").size())
    spots_df["local_area"] = np.array(spots_df["AP_bin"].map(all_surface_areas[stems[k]]["area_normed"]))
    spots_df["local_actual_area"] = np.array(spots_df["AP_bin"].map(all_surface_areas[stems[k]]["areas"]))
    spots_df["AP_bin"] = np.array(spots_df["AP_bin"].astype(float))
    spots_df["nucleus_rel_density"] = spots_df["nucleus_weight"] / spots_df["local_area"]
    spots_df["nucleus_actual_density"] = 1 / spots_df["local_actual_area"]

    """
    Displacement related metrics
    """
    spots_df["time_since_nc10"] = np.abs(spots_df["frame"] - all_mmfs[stem][0])
    idx = spots_df.groupby("track_id")["time_since_nc10"].idxmin()
    result = spots_df.loc[idx].set_index("track_id")["AP"]
    spots_df["track_AP_init"] = spots_df["track_id"].map(result)
    spots_df["displacement_from_start"] = spots_df["AP"] - spots_df["track_AP_init"]
    last_frame = all_mmfs[stem][-1]
    spots_df["final_displacement"] = spots_df["track_id"].map(spots_df[spots_df["frame"] == last_frame].groupby("track_id")["AP"].mean()) - spots_df["AP"]

    """
    Get density and displacement at at min movement frames for each cycle
    """
    min_mvmt_frames = all_mmfs[stem]
    for cycle, frame in zip(cycles, min_mvmt_frames):
        cycle_df = spots_df[spots_df["frame"] == frame].copy()

        positions = cycle_df.groupby("AP_bin")["nucleus_rel_density"].mean().index.values
        densities = cycle_df.groupby("AP_bin")["nucleus_rel_density"].sum().values
        actual_densities = cycle_df.groupby("AP_bin")["nucleus_actual_density"].sum().values
        displacements = cycle_df.groupby("AP_bin")["displacement_from_start"].mean().values
        final_displacements = cycle_df.groupby("AP_bin")["final_displacement"].mean().values
        surface_areas = cycle_df.groupby("AP_bin")["local_actual_area"].mean().values

        cycle_relative_densities["positions"].extend(positions)
        cycle_relative_densities["densities"].extend(densities)
        cycle_relative_densities["actual_densities"].extend(actual_densities)
        cycle_relative_densities["cycle"].extend([cycle for _ in range(len(positions))])
        cycle_relative_densities["count"].extend(cycle_df.groupby("AP_bin")["nucleus_weight"].count().values)
        cycle_relative_densities["area"].extend(cycle_df.groupby("AP_bin")["local_actual_area"].mean().values)
        cycle_relative_densities["condition"].extend([condition_map[stems[k][:-6]] for _ in range(len(positions))])
        cycle_relative_densities["source"].extend([k for _ in range(len(positions))])
        cycle_relative_densities["avg_displacement"].extend(displacements)
        cycle_relative_densities["avg_final_displacement"].extend(final_displacements)
        cycle_relative_densities["surface_area"].extend(surface_areas)


cycle_relative_densities = pd.DataFrame(cycle_relative_densities)
print(cycle_relative_densities)

In [ ]:
print(cycle_relative_densities["positions"].unique())

In [ ]:
def get_plotting_df(relative_densities_df):
    # remove most anterior and posterior positions (pole cells)
    plotting_df = relative_densities_df.query("positions > 0.01 and positions < 0.95").copy()

    return plotting_df

def plot_compare_conditions_at_cycle(relative_densities_df, cycle, conditions, ax, legend=False):
    plotting_df = get_plotting_df(relative_densities_df)

    filtered_df = plotting_df.query("cycle == @cycle and condition in @conditions")

    sns.barplot(filtered_df, x="positions", y="densities", hue="condition", palette=condition_pal, lw=1, legend=legend, ax=ax, edgecolor="k", errorbar=None)
    sns.stripplot(filtered_df, x="positions", y="densities", hue="condition", palette="dark:k", legend=False, dodge=True, ax=ax)

    xtick_positions = ax.get_xticks()
    ax.set_xticks(xtick_positions, ["Anterior", "", "Middle", "", "Posterior"])


def plot_compare_cycles_at_condition(relative_densities_df, condition, cycles, pal, legend=False, style="-"):
    plotting_df = get_plotting_df(relative_densities_df)

    filtered_df = plotting_df.query("condition == @condition and cycle in @cycles")

    sns.barplot(filtered_df, x="positions", y="densities", hue="cycle", palette=pal, lw=1, legend=legend, ax=ax, linestyle=style, errorbar=None, edgecolor="k", alpha=1.0)
    sns.stripplot(filtered_df, x="positions", y="densities", hue="cycle", palette="dark:k", legend=False, dodge=True, ax=ax)

    xtick_positions = ax.get_xticks()
    ax.set_xticks(xtick_positions, ["Anterior", "", "Middle", "", "Posterior"])

fig, ax = plt.subplots(1, 1, figsize=(3, 2.5))
cycles = [10, 14]
condition = "wt"
cycle_pal = {c: color for c, color in zip(cycles, condition_pal_map[condition][::4])}
plot_compare_cycles_at_condition(cycle_relative_densities.query("source != 3"), condition, cycles, {10: "#A2ABB9", 14: "#607380"}, legend=False)
plt.xlabel("")
plt.ylabel("")
plt.ylim(0.5, 1.25)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / f"{condition}_cycle_{cycles[0]}_vs_{cycles[1]}_density_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(4.5, 2.5), sharey=True)
plt.tight_layout()
cycles = [10, 14]

for cycle, ax in zip(cycles, axes):
    condition = "wt"
    cycle_pal = {c: color for c, color in zip(cycles, condition_pal_map[condition][::4])}
    plot_compare_cycles_at_condition(cycle_relative_densities.query("source != 3"), condition, [cycle], {10: "#ee9b00", 14: "#ae2012"}, legend=False)
    ax.set_ylim(0.5, 1.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("")
fig.supxlabel("")
axes[0].set_ylabel("Relative density")
# axes[1].spines[["left"]].set_visible(False)
# axes[1].set_yticks([])
axes[1].set_ylabel("Relative density")
plt.savefig(save_path / f"{condition}_cycle_{cycles[0]}_vs_{cycles[1]}_density_comparison_sideby_side.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(2.5, 3.))
cycles = [10, 14]
condition = "wt"
cycle_pal = {c: color for c, color in zip(cycles, condition_pal_map[condition][::4])}
plot_compare_cycles_at_condition(cycle_relative_densities.query("source != 3"), condition, [10], {10: "#ee9b00", 14: "#ae2012"}, legend=False)
plt.xlabel("AP position")
plt.ylabel("Relative density")
plt.ylim(0.5, 1.25)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / f"{condition}_cycle_{10}_density_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(2.5, 3.))
cycles = [10, 14]
condition = "wt"
cycle_pal = {c: color for c, color in zip(cycles, condition_pal_map[condition][::4])}
plot_compare_cycles_at_condition(cycle_relative_densities.query("source != 3"), condition, [14], {10: "#ee9b00", 14: "#ae2012"}, legend=False)
plt.xlabel("AP position")
plt.ylabel("Relative density")
plt.ylim(0.5, 1.25)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(save_path / f"{condition}_cycle_{14}_density_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

for cycle in [10, 12, 14]:
    fig, ax = plt.subplots(1, 1, figsize=(3.76, 2.46))
    conditions = ["wt", "trk",]
    plot_compare_conditions_at_cycle(cycle_relative_densities, cycle, conditions, ax, legend=False)
    plt.xlabel("AP position")
    plt.ylabel("Relative density")
    # plt.title(f"Cycle {cycle}")
    ax.spines[["top", "right"]].set_visible(False)
    plt.ylim(0.5, 1.25)
    plt.savefig(save_path / f"nc_{cycle}_density_{"".join(conditions)}_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()


fig, axes = plt.subplots(3, 2, figsize=(6, 9), sharey=True)
plt.tight_layout()
for cycle, axrow in zip([10, 12, 14], axes):
    axrow[0].set_ylabel("Relative density")
    for condition, ax in zip(["wt", "trk"], axrow):
        plot_compare_conditions_at_cycle(cycle_relative_densities.query("~source.isin([3])"), cycle, [condition], ax, legend=False)
        ax.set_ylim(0.5, 1.25)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel("")

plt.savefig(save_path / f"nc_{cycle}_density_{"".join(conditions)}_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(6, 4), sharey=True, sharex=True)
plt.tight_layout()
for cycle, axrow in zip([10, 12, 14], axes.T):
    # axrow[0].set_ylabel("Relative density")
    for condition, ax in zip(["wt", "trk"], axrow):
        plot_compare_conditions_at_cycle(cycle_relative_densities.query("~source.isin([3])"), cycle, [condition], ax, legend=False)
        ax.set_ylim(0.5, 1.25)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel("")

plt.savefig(save_path / f"nc_{cycle}_density_{"".join(conditions)}_comparison_horizontal.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(5, 2, figsize=(6, 12), sharey=True)
plt.tight_layout()
for cycle, axrow in zip([10, 11, 12, 13, 14], axes):
    axrow[0].set_ylabel("Relative density")
    for condition, ax in zip(["wt", "trk"], axrow):
        plot_compare_conditions_at_cycle(cycle_relative_densities.query("~source.isin([3])"), cycle, [condition], ax, legend=False)
        ax.set_ylim(0.5, 1.25)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlabel("")

plt.savefig(save_path / f"nc_{cycle}_density_{"".join(conditions)}_comparison_vertical.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import napari

k = 4

df = spots_dfs[k]
print(stems[k])

viewer = napari.Viewer(ndisplay=3)
colors = [cc.glasbey_cool[tid % 255] for tid in df["track_id"]]
viewer.add_points(df[["frame", "z", "y", "x"]], face_color=colors, properties=df["track_id"].values, size=df["radius"]*3, border_color="k")
napari.run()


In [ ]:
for k in range(3):
    df = spots_dfs[k]

    for cycle, frame in enumerate(all_mmfs[stems[k]], 10):
        frame_df = df[df["frame"] == frame]
        print(stems[k], cycle, len(frame_df))
        if cycle == 14:
            print(frame_df["y"].max() - frame_df["y"].min())

In [ ]:
all_transfer_matrices = []

k = 0

print(stems[k])

df = spots_dfs[k].query("AP_bin > 0.001 and AP_bin <= 1.0").copy()


initial_frame = all_mmfs[stems[k]][0]
final_frame = all_mmfs[stems[k]][-1]
surface_area = all_surface_areas[stems[k]]

print(cycle_relative_densities.query("source == @k and cycle == 14")["densities"].values)



# print(surface_area)

# print(df.query("frame == @initial_frame").groupby("AP_bin")["AP"].count())
print(df.query("frame == @final_frame").groupby("AP_bin")["AP"].count().to_numpy())

initial_vec = df.query("frame == @initial_frame").groupby("AP_bin")["AP"].count().to_numpy()[:-1]

print(df.query("frame == @initial_frame").groupby("AP_bin")["AP"].count())

final_frame_df = df.query("frame == @final_frame").copy()
initial_frame_df = df.query("frame == @initial_frame").copy()

final_frame_df["initial_AP_bin"] = final_frame_df["track_id"].map(initial_frame_df.set_index("track_id")["AP_bin"])

print(final_frame_df.groupby("AP_bin")["frame"].count().to_numpy() / initial_vec[-1])

transfer_matrix = final_frame_df.groupby("AP_bin")["initial_AP_bin"].value_counts().unstack().fillna(0)

print(initial_vec)

print(transfer_matrix.iloc[:, :].to_numpy() / initial_vec)

print((transfer_matrix.iloc[:, :].to_numpy() / initial_vec).sum(axis=1))

print(transfer_matrix.iloc[:, :].to_numpy() / initial_vec @ initial_vec)

print(final_frame_df.groupby("AP_bin")["initial_AP_bin"].value_counts(dropna=False))

transfer_matrix = transfer_matrix.iloc[:, :].to_numpy()

transfer_matrix_per_nuc = transfer_matrix / initial_vec

annot_labels = []
for row in range(transfer_matrix_per_nuc.shape[0]):
    annot_labels.append([])

    for col in range(transfer_matrix_per_nuc.shape[1]):
        value = transfer_matrix_per_nuc[row, col]
        if value > 0.05:
            annot_labels[-1].append(f"{value:.1f}")
        else:
            annot_labels[-1].append("")

print(annot_labels)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
cbar_fig, cbar_ax = plt.subplots(1, 1, figsize=(0.5, 3))
sns.heatmap(
    transfer_matrix_per_nuc,
    xticklabels=[],
    yticklabels=[],
    linewidths=0.5,
    linecolor="k",
    fmt="",
    annot=annot_labels,
    cmap="Reds",
    cbar_kws={"label": ""},
    ax=ax,
    cbar_ax=cbar_ax,
    square=True)

fig.savefig(save_path / f"{stems[k][:-6]}_ap_bin_transfer_matrix.png", dpi=300, bbox_inches="tight")
cbar_fig.savefig(save_path / f"{stems[k][:-6]}_ap_bin_transfer_matrix_colorbar.png", dpi=300, bbox_inches="tight")
plt.show()






# fig.savefig(save_path / f"{stems[k][:-6]}_ap_bin_transfer_matrix.png", dpi=300, bbox_inches="tight")
# cbar_fig.savefig(save_path / f"{stems[k][:-6]}_ap_bin_transfer_matrix_colorbar.png", dpi=300, bbox_inches="tight")
# plt.show()


In [ ]:
k = 6

print(stems[k])

df = spots_dfs[k].query("AP_bin > 0.02 and AP_bin < 0.98").copy()

initial_frame = all_mmfs[stems[k]][0]
final_frame = all_mmfs[stems[k]][-1]
surface_area = all_surface_areas[stems[k]]

final_frame_df = df.query("frame == @final_frame").copy()
initial_frame_df = df.query("frame == @initial_frame").copy()
final_frame_df["initial_AP_bin"] = final_frame_df["track_id"].map(initial_frame_df.set_index("track_id")["AP_bin"])

import napari

cmap = sns.color_palette("Spectral", as_cmap=True)

viewer = napari.Viewer()
color = np.array([cmap(ap_bin) for ap_bin in final_frame_df["initial_AP_bin"].values])
points = viewer.add_points(final_frame_df[["x", "y", "z"]].values, face_color=color, size=5, name="Initial frame")